In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

def read_bronze():
    # read all csvs from bronze into dataframe
    df = spark.read.csv(
        "/Volumes/workspace/default/bronze/aemo_raw/*.csv",
        header=True,
        inferSchema=True
    )
    return df

def clean(df):
    df = df.filter(F.col("PERIODTYPE") == "TRADE")  # trade intervals only

    # parse settlement date
    df = df.withColumn(
        "settlement_ts",
        F.to_timestamp("SETTLEMENTDATE", "yyyy/MM/dd HH:mm:ss")
    )

    # cast types
    df = df.withColumn("total_demand_mw", F.col("TOTALDEMAND").cast(DoubleType()))
    df = df.withColumn("price_rrp", F.col("RRP").cast(DoubleType()))

    df = df.drop("SETTLEMENTDATE", "TOTALDEMAND", "RRP", "PERIODTYPE")  # drop raw columns

    df = df.dropDuplicates(["settlement_ts", "REGION"])  # drop overlapping times & regions

    df = df.withColumnRenamed("REGION", "region")  # rename after dedup

    df = df.dropna(subset=["settlement_ts", "total_demand_mw"])  # drop nulls

    return df

def write_silver(df):
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save("/Volumes/workspace/default/silver/energy_clean"))
    print("[SILVER] cleaned data written")

def main():
    df_raw = read_bronze()
    print(f"[SILVER] raw row count: {df_raw.count()}")

    df_clean = clean(df_raw)
    print(f"[SILVER] clean row count: {df_clean.count()}")

    write_silver(df_clean)
    df_clean.printSchema()
    df_clean.show(5)

## Transformation

Run the test cell to verify a single batch before running `main` to process all files.

### Testing

In [0]:
df_raw = spark.read.csv(
    "/Volumes/workspace/default/bronze/aemo_raw/2024_01_VIC1.csv",
    header=True,
    inferSchema=True
)
print(f"raw rows: {df_raw.count()}")
df_raw.printSchema()
df_raw.show(3)

df_clean = clean(df_raw)
print(f"clean rows: {df_clean.count()}")
df_clean.printSchema()
df_clean.show(3)

### Running



In [0]:
main()